##**Sonic Workshop: Intro to LLava**

In [ ]:
#installing packages
!pip install -q transformers==4.36.2
!pip install -q bitsandbytes==0.41.3 accelerate==0.25.0
!pip install git+https://github.com/Imageomics/pybioclip

In [ ]:
import torch
import torch.nn.functional as F
from transformers import BitsAndBytesConfig
from bioclip import TreeOfLifeClassifier, Rank
from PIL import Image

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

##LLaVa

In [ ]:
from transformers import pipeline

model_id = "llava-hf/llava-1.5-7b-hf"
pipe = pipeline("image-to-text", model=model_id, model_kwargs={"quantization_config": quantization_config})

#Connect Google Colab to Your Drive Folder to Import Images Directly
##**Important**: Images you upload here will be deleted when the session ends but if you connect to your drive your images are safe

In [ ]:
path=input('Please enter the path of the image: ')
img = Image.open(path)
img

##How to ask LLava?
You can ask questions to LLava through text about the image, just like you do with ChatGPT.

In [ ]:
Question1 = "Whats the weather in san francisco"
prompt = f'USER: <image>\n{Question1}?\nASSISTANT:'
outputs = pipe(img, prompt=prompt, generate_kwargs={"max_new_tokens": 200})
output = outputs[0]['generated_text']


Question2 ="Yes or No answers only, is there an animal in the provided photo?"
prompt = f'USER: <image>\n{Question2}?\nASSISTANT:'
outputs = pipe(img, prompt=prompt, generate_kwargs={"max_new_tokens": 200})
output = outputs[0]['generated_text']
print(outputs)

if  "Yes" in output:
    Question3 ="Is this question dependant on or involve any animal? "+Question1
    prompt = f'USER: <image>\n{Question3}?\nASSISTANT:'
    outputs = pipe(img, prompt=prompt, generate_kwargs={"max_new_tokens": 200})
    output = outputs[0]['generated_text']
    print(output)
    if  "Yes" in output:
        print("BioClip")
        classifier = TreeOfLifeClassifier()
        predictions = classifier.predict(path, Rank.SPECIES)
        prediction = predictions[0]
        common_name = prediction["common_name"]
        print(common_name)
        Question4 ="The animal in the image is a "+common_name+". Use this animal to answer the question "+Question1
        prompt = f'USER: <image>\n{Question4}?\nASSISTANT:'
        outputs = pipe(img, prompt=prompt, generate_kwargs={"max_new_tokens": 200})
        output = outputs[0]['generated_text']
        print(outputs)

    else:
        print("BioClip not needed")
        prompt = f'USER: <image>\n{Question1}?\nASSISTANT:'
        outputs = pipe(img, prompt=prompt, generate_kwargs={"max_new_tokens": 200})
        output = outputs[0]['generated_text']
        print(outputs)

else:
    print("No Animal in Provided photo")